In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# #Importing Model Data
    
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=data['time']
# # parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=data['time'].values/(1e9 * 60); times=times.astype(float);
# Np_str='125e3'
# #Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,140+1))
# # parcel=parcel.isel(time=np.arange(0,140+1))
# res='1km'

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
true_time=data['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
Np_str='1e6'
#Restricts the timesteps of the data from timesteps0 to 140
res='1km'
index_adjust=0
ocean_fraction=0.25


# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [2]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(PlottingFunctions, inspect.isfunction)]
# functions

In [3]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}.h5'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_TEST.h5'
with h5py.File(in_file, 'r') as f:
    # Load the dataset by its name
    A_g = f['A_g'][:]
    A_c = f['A_c'][:]

    # W = f['W'][:]
    # QCQI = f['QCQI'][:]
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

#Making Time Matrix
Nt=len(data['time'])
T = np.broadcast_to(np.arange(Nt)[:, None], A_c.shape)  # shape: (Nt, p)

In [4]:
#READING BACK IN
mins_thresh=5
# mins_thresh=10
dir3=dir+f'Project_Algorithms/Entrainment/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}.h5'
with h5py.File(dir3, 'r') as h5file:
    A_g_Processed = h5file['A_g_Processed'][:]
    A_c_Processed = h5file['A_c_Processed'][:]

In [ ]:
#2D VERSION

In [14]:
def ed2d(A, T, Z, type):
    start_time = time.time()
    """
    Function to compute 2D entrainment and update result array based on provided inputs.
    
    Returns a 2D (t,z) array containing the sum of the D array representing entrained parcels, by 1, and detrained parcels, by -1.
    The finally array is then ordered by the appropiate index using the np.add.at function
    
    Parameters:
    - A: The (t,p) lagrangian binary array.
    - T: The (t,p) lagrangian time index array.
    - Z: The (t,p) Lagrangian z index array.

    """
    # Compute the difference between neighboring elements along the first axis
    D = np.zeros_like(A)
    D[1:, :] = A[1:, :] - A[:-1, :]
    
    # Update D for entrainment/detrainment
    if type=='e':
        D[D <= 0] = 0
    elif type=='d':
        D[D >= 0] = 0
    
    # Initialize time and vertical dimension arrays
    Nt = len(data['time']); Nz = len(data['zh'])
    
    # Initialize result array
    result = np.zeros((Nt, Nz))
    
    # Use np.add.at to accumulate values in the result array
    np.add.at(result, (T, Z), D)
    
    end_time = time.time()
    print(f"Execution time: {(end_time - start_time)} seconds")
    return result

In [15]:
A=A_c_Processed
out=ed2d(A,T,Z,type='e')
t=100
out[t]

Execution time: 11.262207269668579 seconds


array([  0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   7.,  65., 130.,
       458., 294., 262., 216., 210., 174., 164., 116.,  95., 168., 268.,
       325., 352., 454., 481., 465., 476., 329., 288., 213., 105.,  62.,
         0.])

In [16]:
PROCESSING=False
PROCESSING=True
if PROCESSING==False:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_entrainmentdetrainment_profiles_{res}_{Np_str}.h5'
if PROCESSING==True:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_entrainmentdetrainment_profiles_PREPROCESSING_{res}_{Np_str}.h5'
with h5py.File(dir3, "r") as h5f:
    profile_array_e_c = h5f["profile_array_e_c"][:]
profile_array_e_c[t]

array([  0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   7.,  65., 130.,
       458., 294., 262., 216., 210., 174., 164., 116.,  95., 168., 268.,
       325., 352., 454., 481., 465., 476., 329., 288., 213., 105.,  62.,
         0.])

In [17]:
A=A_c_Processed
out=ed2d(A,T,Z,type='d')
t=100
out[t]

Execution time: 11.31309700012207 seconds


array([   0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,  -11.,
        -21.,  -88., -188., -198., -193., -221., -216., -237., -208.,
       -173., -121., -133., -257., -381., -452., -445., -519., -470.,
       -476., -425., -319., -183.,  -92.,  -24.,   -1.])

In [18]:
PROCESSING=False
PROCESSING=True
if PROCESSING==False:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_entrainmentdetrainment_profiles_{res}_{Np_str}.h5'
if PROCESSING==True:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_entrainmentdetrainment_profiles_PREPROCESSING_{res}_{Np_str}.h5'
with h5py.File(dir3, "r") as h5f:
    profile_array_d_c = h5f["profile_array_d_c"][:]
profile_array_d_c[t]

array([  0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,  11.,  21.,  88.,
       188., 198., 193., 221., 216., 237., 208., 173., 121., 133., 257.,
       381., 452., 445., 519., 470., 476., 425., 319., 183.,  92.,  24.,
         1.])

In [ ]:
#3D ALGORITHM

In [11]:
import numpy as np

def ed3d(A, T, Z, Y, X, type):
    start_time = time.time()
    """
    Function to compute 3D entrainment and update result array based on provided inputs.
    
    Returns a 4D (t,z,y,x) array containing the sum of the D array representing entrained parcels, by 1, and detrained parcels, by -1.
    The finally array is then ordered by the appropiate index using the np.add.at function
    
    Parameters:
    - A: The (t,p) lagrangian binary array.
    - T: The (t,p) lagrangian time index array.
    - Z: The (t,p) Lagrangian z index array.
    - Y: The (t,p) Lagrangian y index array.

    """
    # Compute the difference between neighboring elements along the first axis
    D = np.zeros_like(A)
    D[1:, :] = A[1:, :] - A[:-1, :]
    
    # Update D for entrainment/detrainment
    if type=='e':
        D[D <= 0] = 0
    elif type=='d':
        D[D >= 0] = 0
    
    # Initialize time and vertical dimension arrays
    Nt = len(data['time']); Nz = len(data['zh']); Ny = len(data['yh']); Nx = len(data['xh'])
    
    # Initialize result array
    result = np.zeros((Nt, Nz, Ny, Nx))
    
    # Use np.add.at to accumulate values in the result array
    np.add.at(result, (T, Z, Y, X), D)
    
    end_time = time.time()
    print(f"Execution time: {(end_time - start_time)} seconds")
    return result

In [ ]:
A=A_c_Processed
out=ed3d(A,T,Z,Y,X,type='e')

t=100
summed=np.sum(out,axis=(2,3))
summed[t,:]

In [6]:
PROCESSING=False
PROCESSING=True
if PROCESSING==False:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_entrainmentdetrainment_profiles_{res}_{Np_str}.h5'
if PROCESSING==True:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_entrainmentdetrainment_profiles_PREPROCESSING_{res}_{Np_str}.h5'
with h5py.File(dir3, "r") as h5f:
    profile_array_e_c = h5f["profile_array_e_c"][:]
profile_array_e_c[t]

array([  0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   7.,  65., 130.,
       458., 294., 262., 216., 210., 174., 164., 116.,  95., 168., 268.,
       325., 352., 454., 481., 465., 476., 329., 288., 213., 105.,  62.,
         0.])

In [7]:
A=A_c_Processed
out=ed3d(A,T,Z,Y,X,type='d')

t=100
summed=np.sum(out,axis=(2,3))
summed[t,:]

Execution time: 22.159555196762085 seconds


array([   0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,  -11.,
        -21.,  -88., -188., -198., -193., -221., -216., -237., -208.,
       -173., -121., -133., -257., -381., -452., -445., -519., -470.,
       -476., -425., -319., -183.,  -92.,  -24.,   -1.])

In [8]:
PROCESSING=False
PROCESSING=True
if PROCESSING==False:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_entrainmentdetrainment_profiles_{res}_{Np_str}.h5'
if PROCESSING==True:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_entrainmentdetrainment_profiles_PREPROCESSING_{res}_{Np_str}.h5'
with h5py.File(dir3, "r") as h5f:
    profile_array_d_c = h5f["profile_array_d_c"][:]
profile_array_d_c[t]

array([  0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,  11.,  21.,  88.,
       188., 198., 193., 221., 216., 237., 208., 173., 121., 133., 257.,
       381., 452., 445., 519., 470., 476., 425., 319., 183.,  92.,  24.,
         1.])